# SC RAPIDS MULTI GPU TUTOTRIAL

Disclaimer: A standardized notebook is meant to be a template for analysis. You are expected to review the notebook contents and adjust as needed for your own analyses. However, deviations from the standard recommendations should be described and justified in your notebook text.

    

## Environment
The kernel environment for this notebook can be easily set up for hise GPU configuration using the pixi.toml file in this directory by running the command below:<br>
`pixi run kernel` <br>
Sometimes it takes a few minutes, the 'rapids' ipy kernel should be available and selected before moving on.

## Notebook Description
This notebook walks through a single cell rna-seq anaylsis using rapids-singlecell + DASK implementation for decreased runtime and low memory peak.<br>
This workflow expects raw count expression matrix in anndata format within the input zarr storage. Anndata objects can quickly be converted to zarr with Adata.write_zarr() or using our in house tool for speedy conversion https://github.com/aifimmunology/DataScales 

### This notebook:
1. **Bulds GPU cluster on HISE**
2. **Walks through the analysis steps below with function calls optimized for our 4 GPU IDE on HISE**

### Analysis steps
The api for calling analysis functions is nearly the same as scanpy.
API documentation for all supported functions can be found here: https://rapids-singlecell.readthedocs.io/en/latest/api/scanpy_gpu.html#preprocessing-pp <br>
Those included in this analysis:
- normalize_total()
- log1p()
- highly_variable_genes()
- scale()
- pca()
- harmony_integrate() OPTIONAL
- neighbors()
- umap()
- leiden()

## Input
- **Zarr storage holding gene expression data.** Anndata objects can quickly be converted to zarr with Adata.write_zarr() or using our in house tool for speedy conversion https://github.com/aifimmunology/DataScales
- **Analysis Paramters** See section below with recommendations

## Output
- **analysis embeddings** Embeddings are made through this analysis such as, UMAP coordinates, leiden clustering labels, HVG selections + other steps. these can be explored with the object loaded in this notebook, or saved to a file.
- **UMAP visualization** 

In [ ]:
# Variables
data_pth = "/home/workspace/zarrs/5M_sparse_sorted_9.zarr" #zarr, sparse format data works best
ROW_CHUNK_SIZE = 24_000 #Number of cells it pulls per chunk. Tune with Zarr chunking. 24k worked well on 13M, and 5M dataset that was stored in sparse zarr.
RANDOM_SEED = 4242

In [ ]:
# Harmony Usage
BATCH_KEY = [] #List str:  Change to the obs columns that you would like to integrate across. If this is empty, Harmony will not be run

## 1. Create CUDA DASK cluster
Building out the GPU cluster. This is how rapids will work with the GPU set.

### Customizability
**Threads_per_worker**: Can significantly speed up cpu processing steps at cost of a bit of peak memory. 16 worked well for 5M dataset<br>
**CUDA_VISIBLE_DEVICES**: Starting with 0, set with list of all gpus. in this case we have 4. Multi-gpu can help a small amount, single gpu works great as well.

This example configuration is robust against memory errors and crashes. See https://rapids-singlecell.readthedocs.io/en/latest/out_of_core.html for other configurations


### Initial Imports

In [ ]:
import dask
import time
import warnings
import gc

from dask_cuda import LocalCUDACluster
from dask.distributed import Client

### Multi-GPU and single GPU guidelines
For this workflow, we will utilize a 4 gpu machine, [0,1,2,3]. Multi-GPU can accelerate a few of the initial steps, but single-gpu usage sees similar runtimes. 
- **To see in-depth benchmarks for speed, memory, dataset-size, visit my rapids benchmarking layout [https://github.com/aifimmunology/DataScales/tree/main/benchmarks/rapids_singlecell](https://github.com/aifimmunology/DataScales/tree/main/benchmarks/rapids_singlecell)**
- A single GPU instance with 15G of VRAM ran a 5 Million cell dataset in around 15 minutes, while the multi-GPU instance shaved a few minutes off.
- This dask-cuda setup is configured to be robust with memory usage, spilling into host RAM when GPU fills up. This has run fine up to 13M cell dataset. Beyond that, it is recomended to get onto a GPU with more than 30GB of internal VRAM.

To see the number of GPUs and their memory for your current instance, run the cell below.

In [ ]:
%%bash
nvidia-smi

In [ ]:
%%time
# needed so dask doesnt time out on steps its not used
dask.config.set({"distributed.scheduler.worker-ttl": None})

cluster = LocalCUDACluster(
    CUDA_VISIBLE_DEVICES="0,1,2,3",  #change here to whats needed
    protocol="tcp", 
    threads_per_worker=16,
    rmm_managed_memory=True, #allowing spilling into RAM from gpu
    rmm_allocator_external_lib_list="cupy",
    enable_cudf_spill=True,
)

client = Client(cluster)


In [ ]:
import rapids_singlecell as rsc
import anndata as ad

from anndata.experimental import read_elem_lazy as read_dask
import zarr
zarr.config.set({'async.concurrency': 10, 'threading.max_workers': 4, })

#reinit to get managed memory on. Required in documentation?
import rmm
import cupy as cp
from rmm.allocators.cupy import rmm_cupy_allocator
rmm.reinitialize(managed_memory=True, pool_allocator=False)
cp.cuda.set_allocator(rmm_cupy_allocator)

# Load data from zarr file structure
This step loads the zarr into a dask graph. The dask graph holds the location of chunks of size CELL_CHUNK_SIZE x all genes. And instructions on how to grab it and what computation to run on it quickly. This perfectly aligns with zarr (especially if zarr and CELL_CHUNK_SIZE chunks are similar sizes) for quick and parallel access to data.
## Set input data path below

In [ ]:
import numpy as np

f = zarr.open(data_pth, mode = 'a')
X = f["X"]
shape = X.attrs["shape"]
print("Matrix Shape: ", shape) #Checking shape of matrix and data type to ensure raw count data
dtype = X['data'] if hasattr(X,'keys') and 'data' in X else X
print("Data Type:", dtype.dtype, " - Should be int* or uint* for raw counts. Else its already normalized")

#Create dask array
X_dask = read_dask(X, (ROW_CHUNK_SIZE, shape[1]))
if np.issubdtype(X_dask.dtype, np.integer):
    X_dask = X_dask.astype(np.float32)
    
adata = ad.AnnData(
    X = X_dask,
    obs = ad.io.read_elem(f["obs"]),
    var = ad.io.read_elem(f["var"])
)

rsc.get.anndata_to_GPU(adata) #lets dask know to load adata to gpu, when it is needed. For now this does no IO.

## Preprocessing
If the datatype is an integer, can safely assume it is the raw data, and move forward with normalizing and log transforming

In [ ]:
%%time
if 'int' in str(dtype.dtype):
    rsc.pp.normalize_total(adata)
    rsc.pp.log1p(adata)

## Highly Variable Gene Selection
The only flavor that plays well with dask is seurat, used here. <br>
<u>The full expression matrix after normalization is saved here to 'adata_full'. This is whats written to the output file</u>
### Persisting
We are persisting the adata object to GPU memory after signigicant steps. This keeps it in GPU memory which speeds up the next steps significantly. A few steps including HVG use this persist() capability.

In [ ]:
%%time
rsc.pp.highly_variable_genes(
    adata, flavor="seurat", n_top_genes=2000,
)

import numpy as _np_hvg
hvg_mask = _np_hvg.asarray(adata.var["highly_variable"])
adata = adata[:, hvg_mask].copy()

n_rows, n_cols = adata.shape[0], adata.shape[1]
rows_per_worker = (n_rows+4-1)//4
adata.X = adata.X.rechunk((rows_per_worker, n_cols)).persist()
adata.X.compute_chunk_sizes()

## Scaling & PCA
Scaling zero_center = False, this avoids altering all the 0 entries (very many). Use for sparse format data. Can turn =True for dense.<br>
<br>
PCA svd_solver uses covariance_eigh by default, this is set to highlight that. Dask best uses this. Persisting after PCA for future steps.

In [ ]:
%%time
adata.X = adata.X.astype("float64")  #Helped with rounding accuracy in runs, only for scaling step, default it is float32
rsc.pp.scale(adata, zero_center=False, max_value=10) #Zero_center set to true is your zarr is not sparse format.

In [ ]:
%%time
warnings.filterwarnings("ignore", category=UserWarning)
rsc.pp.pca(adata, n_comps=50, random_state=RANDOM_SEED)

adata.obsm["X_pca"]=adata.obsm["X_pca"].persist()
adata.obsm["X_pca"].compute_chunk_sizes()
adata.obsm["X_pca"]=adata.obsm["X_pca"].compute()

## Harmony integration
Harmony integration over selected adata.obs columns. List the obs to integrate on in BATCH_KEY list.<br> Example here integrates on 'pool_id'

In [ ]:
%%time
rep = "X_pca"
if BATCH_KEY :
    adata.obs[BATCH_KEY] = adata.obs[BATCH_KEY].astype("category")
    rsc.pp.harmony_integrate(
        adata, key=BATCH_KEY,
        basis="X_pca", adjusted_basis="X_pca_harmony",
        random_state=RANDOM_SEED
    )
    rep = "X_pca_harmony"

## Neighbors
Neighbors can build exact neighbors graph using algorithm = 'brute' (rapids default). <br>
For large datasets and smaller GPUS. 'mg_ivfflat' is a fast algorithm that approximates the neighbors graph on multi-gpu, similar to scanpy implementation.<br>
**'Brute' took over 30 minutes on a 5M dataset, while 'mg_ivfflat' took ~ 1 minute.**

In [ ]:
%%time
rsc.pp.neighbors(
    ##'brute' alg best but long, 'mg_ivfflat' for multi-gpu, or 'ivfflat' for single gpu.
    adata, n_neighbors=20, n_pcs=30, use_rep=rep, algorithm="mg_ivfflat", random_state=RANDOM_SEED 
)

## UMAP
**Init_pos:** Spectral is default, this can take longer to run but gives consistent and better looking results. Can use 'random' for shorter/new visualizations <br> This builds the visualization cordinates, for viewing UMAP, see cell after leidens step.

In [ ]:
%%time
rsc.tl.umap(adata, min_dist=0.45, init_pos='spectral', n_components=2, random_state=RANDOM_SEED)

## Leiden
The resolution showed slightly more cluster compared to scanpy equivilent. Resolution 1.1 ~ 1 on scanpy. N_iterations = 100 is default for rapids, compared to 2 on scanpy.<br>
**use_dask** can be turned to True for large datasets > 10M cells. Uses multi-gpu capabilities.

In [ ]:
%%time
rsc.tl.leiden(adata, resolution=1.1, n_iterations=100, random_state=RANDOM_SEED)#, use_dask=True)

In [ ]:
print("# of clusters: ", len(adata.obs['leiden'].cat.categories), "\nCells per cluster")
print(adata.obs['leiden'].value_counts())

## UMAP visualization with clusters
Visualize select labels on the umap

In [ ]:
import scanpy as sc

rsc.get.anndata_to_CPU(adata)#bring back to CPU
candidate_cols = ["leiden", "aifi_label_l1", "aifi_label_l2", 'pool_id'] #Can add other obs columns here for batch effect checks.

sc.pl.umap(adata, color=candidate_cols, size=2, legend_loc="none")

## Tuning rapids-singlecell
Find benchmark results between different configurations, and dataset sizes at [DataScales repo Benchmark](https://github.com/aifimmunology/DataScales/tree/main/benchmarks) <br>
Benchmarking tool for rapids lives here to quickly run different GPU clusters, and rapids arguments to get per-step benchmark reports

### Package version info

In [ ]:
import sys, importlib.metadata as meta

print(f"Python: {sys.version.split()[0]}")
for pkg in ["anndata", "zarr", "rapids-singlecell-cu13", "numpy", "pandas"]:
    try:    print(f"{pkg}: {meta.version(pkg)}")
    except: print(f"{pkg}: not found")

## Write output
For appending back to the zarr file. 'f' is the open zarr store in this workflow <br>
'final_labels' here would represent the finished cell labels, similar to leidens output

In [ ]:
f["obsm/X_umap"] = adata.obsm["X_umap"]

#writing a obs columns such as cell labels
labels = adata.obs["final_labels"]
ad.io.write_elem(root["obs"], "final_labels", labels.values)

#Needed to update the zarr metadata on the new obs column
order = list(root["obs"].attrs.get("column-order", []))
if "final_labels" not in order:
    root["obs"].attrs["column-order"] = [*order, "final_labels"]